# AlphaFold DB Fetch — Example

![AlphaFold DB Fetch — Example](https://proto-bio.github.io/proto-assets/images/tool/alphafold_db/hero.png)

This notebook demonstrates `run_alphafold_db_fetch`, which retrieves an AlphaFold prediction record from the [AlphaFold Protein Structure Database](https://alphafold.ebi.ac.uk/) by UniProt accession. It returns the predicted structure (PDB or mmCIF), per-residue pLDDT confidence scores, and optionally the predicted aligned error (PAE) matrix, alongside metadata such as gene name, organism, model version, and source URLs. See the AlphaFold DB paper: Varadi et al., *Nucleic Acids Research* (2022), [doi:10.1093/nar/gkab1061](https://doi.org/10.1093/nar/gkab1061).

In [1]:
from proto_tools.tools.database_retrieval import (
    AlphaFoldDBFetchConfig,
    AlphaFoldDBFetchInput,
    run_alphafold_db_fetch,
)
from proto_tools.utils.notebook_docs import display_api_reference

## Input API Reference

In [2]:
display_api_reference("alphafold_db", "input")

**Input** — `AlphaFoldDBFetchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>uniprot_id</code> | <code>str</code> | required | UniProt accession (e.g. 'P04637') |
| <code>isoform</code> | <code>int &#124; None</code> | <code>None</code> | Isoform number to fetch; None returns the canonical record. For non-canonical isoforms only. |

## Config API Reference

In [3]:
display_api_reference("alphafold_db", "config")

**Config** — `AlphaFoldDBFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>structure_format</code> | <code>Literal['pdb', 'cif']</code> | <code>'pdb'</code> | Structure file format (pdb or mmCIF); ignored when include_structure=False |
| <code>include_structure</code> | <code>bool</code> | <code>True</code> | Fetch the structure body and per-residue pLDDT; disable for metadata-only probes |
| <code>include_pae</code> | <code>bool</code> | <code>False</code> | Also fetch the predicted aligned error matrix; tens of MB for long proteins |
| <code>include_msa</code> | <code>bool</code> | <code>False</code> | Fetch the A3M MSA used for prediction; large for conserved proteins |

## Output API Reference

In [4]:
display_api_reference("alphafold_db", "output")

**Output** — `AlphaFoldDBFetchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>uniprot_accession</code> | <code>str</code> | required | Primary UniProt accession |
| <code>entry_id</code> | <code>str</code> | required | AlphaFold entry identifier (e.g. 'AF-P04637-F1') |
| <code>gene</code> | <code>str &#124; None</code> | <code>None</code> | Gene symbol |
| <code>organism_scientific_name</code> | <code>str &#124; None</code> | <code>None</code> | Source organism scientific name |
| <code>tax_id</code> | <code>int &#124; None</code> | <code>None</code> | NCBI taxonomy ID |
| <code>sequence</code> | <code>str</code> | required | Amino-acid sequence covered by the prediction |
| <code>sequence_length</code> | <code>int</code> | required | Length of the predicted sequence |
| <code>sequence_start</code> | <code>int</code> | required | 1-indexed inclusive start residue of the prediction |
| <code>sequence_end</code> | <code>int</code> | required | 1-indexed inclusive end residue of the prediction |
| <code>latest_version</code> | <code>int</code> | required | Latest AlphaFold DB version of this prediction (always the served version) |
| <code>model_created_date</code> | <code>str &#124; None</code> | <code>None</code> | ISO 8601 model creation timestamp |
| <code>mean_plddt</code> | <code>float &#124; None</code> | <code>None</code> | Mean per-residue pLDDT confidence (0-100 scale) |
| <code>pdb_url</code> | <code>str</code> | required | URL to PDB structure file |
| <code>cif_url</code> | <code>str</code> | required | URL to mmCIF structure file |
| <code>bcif_url</code> | <code>str &#124; None</code> | <code>None</code> | URL to BinaryCIF structure file (None on legacy entries) |
| <code>pae_doc_url</code> | <code>str</code> | required | URL to PAE JSON document |
| <code>plddt_doc_url</code> | <code>str</code> | required | URL to per-residue pLDDT JSON document |
| <code>pae_image_url</code> | <code>str</code> | required | URL to rendered PAE PNG |
| <code>msa_url</code> | <code>str &#124; None</code> | <code>None</code> | URL to MSA A3M file, when present |
| <code>am_annotations_url</code> | <code>str &#124; None</code> | <code>None</code> | URL to AlphaMissense pathogenicity CSV (sequence coords) |
| <code>am_annotations_hg19_url</code> | <code>str &#124; None</code> | <code>None</code> | URL to AlphaMissense annotations on GRCh37 |
| <code>am_annotations_hg38_url</code> | <code>str &#124; None</code> | <code>None</code> | URL to AlphaMissense annotations on GRCh38 |
| <code>sequence_checksum</code> | <code>str &#124; None</code> | <code>None</code> | CRC64 checksum of the predicted sequence (cache validation) |
| <code>structure</code> | <code>Structure &#124; None</code> | <code>None</code> | Parsed AlphaFold Structure (PLDDT B-factors, metrics with pLDDT and PAE); None when skipped |
| <code>msa_a3m</code> | <code>str &#124; None</code> | <code>None</code> | A3M-format MSA contents used as input to prediction |
| <code>source_url</code> | <code>str</code> | required | AlphaFold DB API URL used for lookup |
| <code>raw_entry</code> | <code>dict[str, Any]</code> | <code>{}</code> | Complete AlphaFold DB JSON record for advanced programmatic access |

**Metrics** — `AlphaFoldDBMetrics`

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>avg_plddt</code> **(primary)** | <code>float</code> | <code>[0, 100]</code> |  |  |
| <code>plddt_per_residue</code> | <code>list[float]</code> | <code>[0, 100]</code> |  |  |
| <code>pae</code> | <code>list[list[float]]</code> | <code>&gt;= 0</code> |  |  |

In [5]:
# Basic usage: fetch the AlphaFold prediction for human TP53 (UniProt P04637)
# with all defaults (PDB structure + per-residue pLDDT, PAE off).
inputs = AlphaFoldDBFetchInput(uniprot_id="P04637")
config = AlphaFoldDBFetchConfig()

output = run_alphafold_db_fetch(inputs, config)

print(f"UniProt accession: {output.uniprot_accession}")
print(f"AlphaFold entry id: {output.entry_id}")
print(f"Sequence length:   {output.sequence_length}")
print(f"Mean pLDDT:        {output.mean_plddt}")

print(f"\nStructure format:  {output.structure.structure_format}")
print(f"B-factor type:     {output.structure.b_factor_type.value}")
print("\nStructure file (first 80 chars):")
print(output.structure.structure[:80])

plddt = output.structure.metrics["plddt_per_residue"]
print(f"\nstructure.metrics keys:   {sorted(output.structure.metrics.keys())}")
print(f"per-residue pLDDT length: {len(plddt)}")
print(f"first 5 pLDDT values:     {plddt[:5]}")

UniProt accession: P04637
AlphaFold entry id: AF-P04637-F1
Sequence length:   393
Mean pLDDT:        75.06

Structure format:  pdb
B-factor type:     pLDDT

Structure file (first 80 chars):
HEADER                                            01-AUG-25                     

structure.metrics keys:   ['avg_plddt', 'plddt_per_residue']
per-residue pLDDT length: 393
first 5 pLDDT values:     [42.72, 43.59, 37.78, 45.91, 39.31]


In [6]:
# With the PAE matrix: useful for assessing inter-domain confidence.
# PAE files can be large for long proteins, so this is opt-in.
pae_output = run_alphafold_db_fetch(
    AlphaFoldDBFetchInput(uniprot_id="P04637"),
    AlphaFoldDBFetchConfig(include_pae=True),
)

pae = pae_output.structure.metrics["pae"]
print(f"PAE matrix shape: ({len(pae)}, {len(pae[0])})")

print("\nTop-left 3x3 corner (angstroms):")
for row in pae[:3]:
    print([round(v, 3) for v in row[:3]])

PAE matrix shape: (393, 393)

Top-left 3x3 corner (angstroms):
[0.0, 1.0, 3.0]
[2.0, 0.0, 1.0]
[3.0, 2.0, 0.0]


In [7]:
# Metadata-only fetch: skip the structure download.
# This is the cheapest call when you only need URLs and identifiers.
meta_output = run_alphafold_db_fetch(
    AlphaFoldDBFetchInput(uniprot_id="P04637"),
    AlphaFoldDBFetchConfig(include_structure=False),
)

print(f"PDB URL:        {meta_output.pdb_url}")
print(f"mmCIF URL:      {meta_output.cif_url}")
print(f"pLDDT doc URL:  {meta_output.plddt_doc_url}")
print(f"PAE doc URL:    {meta_output.pae_doc_url}")
print(f"Mean pLDDT:     {meta_output.mean_plddt}")
print(f"structure is None: {meta_output.structure is None}")

PDB URL:        https://alphafold.ebi.ac.uk/files/AF-P04637-F1-model_v6.pdb
mmCIF URL:      https://alphafold.ebi.ac.uk/files/AF-P04637-F1-model_v6.cif
pLDDT doc URL:  https://alphafold.ebi.ac.uk/files/AF-P04637-F1-confidence_v6.json
PAE doc URL:    https://alphafold.ebi.ac.uk/files/AF-P04637-F1-predicted_aligned_error_v6.json
Mean pLDDT:     75.06
structure is None: True


In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("alphafold_db_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'alphafold_db_results.json'}")

Exported to example_output/alphafold_db_results.json
